# CopyLane 판정 인코더 — 실험 카드 9 : 정상 합성 문구를 음성으로 더한다 (박수진, 2026-10-08)

> Baseline = v10 (`copylane-encoder-kcbert-v10-합성O-이유O`) · 골든 `529c970556d7` · 공통 dev 749행 (지문 `4ebfcb231da703b5`)
> 규약 `docs/psj/notes/인코더_개선실험_규약_20261007.md` · 카드 원문 `docs/psj/reports/v10_문턱규칙_거짓과장_실험기록_20261007.md` 카드 9 (**돌리기 전에 올렸다**)
> 🔴 **dev 만 본다. 이 노트북에는 test 채점 셀이 없다** (D-175).

## 무엇을 하나

권소라의 카피 생성기가 만든 **정상(적법) 광고 문구 900건**(식품 · 건기식 · 화장품 각 300 · 제품 45종)에 인코더가 76% 꼴로 후보를 낸다(주로 거짓_과장).
dev 에는 적법 광고형 문장이 없어 그동안 재지 못한 과탐이다. 이 문구를 **유형 없는 음성**으로 학습에 더하면 줄어드는지 본다.

| 실험 | Baseline 과 다른 점 (하나) | 겨냥한 지표와 목표 (돌리기 전에 적음) |
|---|---|---|
| `Baseline재현` | 없음 — v10 과 같은 학습 (정상 합성 문구에서의 수를 재려고 다시 학습한다) | — |
| **카드 9** `카드9-정상음성` | 학습에 정상 합성 문구 569행을 음성으로 더한다 (3,964 → 4,533행) | 떼어 둔 정상 합성 문구 285행 중 후보가 서는 비율이 **Baseline 재현의 절반 이하** |

- 정상 합성 문구 900건을 **제품 단위로** 나눈다 — 품목마다 제품 5종(시드 42)을 떼어 측정용(「합성 정상 dev」)으로 두고 나머지 30종을 학습에 쓴다.
- **애매 표시 46건은 학습에도 목표에도 넣지 않는다.** 따로 재서 참고로만 본다.
- 잃어도 되는 것은 통과 기준 네 개(B · G1 · G2 · G3) 그대로다 — 공통 dev 로 잰다.
- 🚨 이 문구는 **사람이 판독하지 않았다**(판독 칸이 전부 비어 있다). 「적법일 것」으로 생성한 문장을 그대로 음성으로 쓴 탐색 실험이다.

## 판정은 어디서 하나

- 통과 기준은 **공통 채점 스크립트**로 판정한다 — 시드별 dev 행별 확률 CSV 를 받아 `score_encoder_dev.py` 에 넣는다 (섹션 8).
- 목표(합성 정상 dev)는 이 노트북의 표 4 가 낸다. 시드별 **합성 정상 행별 확률 CSV** 도 함께 남기므로 다시 셀 수 있다.

## 실행 순서

1. 런타임 유형 **GPU(T4)** → 위에서부터 끝까지 실행. `golden.jsonl` 과 정상 합성 문구 CSV 를 올린다.
2. 학습은 2 실험 × 시드 3 = **6번** (T4 에서 약 30분). 시드 하나가 끝날 때마다 Drive `copylane_v10_card9/` 에 기록한다 — 런타임이 끊겨도 다시 실행하면 **끝난 시드는 건너뛴다**.
3. 실험마다 **시드별 dev 행별 확률 CSV 3개 · 합성 정상 행별 확률 CSV 3개 · 문턱 zip** 이, 카드 9 는 **모델 zip** 까지 Drive 에 남는다 (**저장소에는 올리지 않는다** · D-249).

## 1. 환경 설정 + 파일 업로드 (골든 · 정상 합성 문구 CSV)

`sympy` 충돌로 `from transformers import ...` 가 깨지면 런타임 > 세션 다시 시작 후 첫 셀은 건너뛰고 다음 셀부터 실행한다.

In [ ]:
!pip install -q -U sympy transformers accelerate

In [ ]:
import os, json, hashlib, shutil, gc, math, random, csv, time, zipfile
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
from IPython.display import display

from google.colab import files as colab_files
if not os.path.exists("golden.jsonl"):
    print("① golden.jsonl 선택")
    colab_files.upload()
SYN_NAME = "LLM_카피생성_정상카피_900_판독용_ksr_2026-10-06.csv"
if not os.path.exists(SYN_NAME):
    print("② 정상 합성 문구 CSV 선택 —", SYN_NAME)
    for _f in colab_files.upload():                  # 브라우저가 이름을 바꿔 올렸으면(「 (1)」 등) 제 이름으로 둔다
        if _f.endswith(".csv") and _f != SYN_NAME:
            os.replace(_f, SYN_NAME)

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

GOLDEN_SHA = sha256("golden.jsonl")
print("golden.jsonl sha256:", GOLDEN_SHA[:16], "…")
# D-176: 입력 지문을 **대조**한다
EXPECTED_GOLDEN_SHA = "529c970556d77bd2c231526e20531294bf469f794e74a4796835a36f456eb7fe"   # 재동결 10-05 (8,383행)
EXPECTED_ROWS = (8383, 5894, 2489)   # 전체 · 학습 · 평가
if GOLDEN_SHA != EXPECTED_GOLDEN_SHA:
    raise SystemExit(f"🔴 golden sha 가 원장과 다르다 — 받은 {GOLDEN_SHA[:16]} · 원장 {EXPECTED_GOLDEN_SHA[:16]} (D-176). "
                     "런타임에 예전 파일이 남았으면 런타임을 새로 시작한다")
SYN_SHA = sha256(SYN_NAME)
EXPECTED_SYN_SHA = "aae3b4a5ab5471d31583ca704e0e4fc9bfc24b56693b65e13569a759ecd9cd22"      # 권소라 10-06 판 (900행)
if SYN_SHA != EXPECTED_SYN_SHA:
    raise SystemExit(f"🔴 정상 합성 문구 CSV 가 카드에 적은 판과 다르다 — 받은 {SYN_SHA[:16]} · 기대 {EXPECTED_SYN_SHA[:16]}. "
                     "판독 칸을 채운 새 판이면 새 카드로 돌린다")
print("정상 합성 문구 CSV sha256:", SYN_SHA[:16], "…")

## 2. 스위치 — 바꿀 것은 이 셀뿐이다

| 스위치 | 기본값 | 뜻 |
|---|---|---|
| `RUNS` | Baseline 재현 · 카드 9 | 돌릴 실험. Baseline 재현은 합성 정상 dev 의 기준 수를 내려고 함께 돌린다 |
| `SAVE_MODEL_ZIP` | `True` | 카드 9 의 가장 좋은 시드의 모델 zip(약 420MB)을 Drive 에 둔다. Baseline 재현의 모델은 저장하지 않는다 |
| `SMOKE` | `False` | `True` = 1분짜리 점검(시드 1 · 1 epoch · 학습 300줄). 수치는 의미 없다 |
| `SYN_HOLD_PER_CAT` · `SYN_SEED` | 5 · 42 | 🔒 품목마다 측정용으로 떼는 제품 수 · 뽑는 시드. **카드에 적은 값이다. 고치지 않는다** |
| `GATE` · `TARGET` | 아래 값 | 🔒 **규약 · 실험 카드에 적은 값이다. 고치지 않는다** |

In [ ]:
RUN_DEFS = {                      # 이름 → 정상 합성 문구를 음성으로 더하는가 (그 밖은 Baseline 과 같다)
    "Baseline재현":    {"syn": False},     # v10 과 같은 학습
    "카드9-정상음성":  {"syn": True},      # + 정상 합성 문구(애매 표시 제외 · 떼어 둔 제품 제외)를 유형 없는 음성으로
}
RUNS = ["Baseline재현", "카드9-정상음성"]
BASE_RUN = "Baseline재현"
SAVE_MODEL_ZIP = True
SMOKE = False

# ── 학습 설정 — Baseline(v10)과 같다. 바꾸지 않는다
MODEL_NAME = "beomi/kcbert-base"
SEEDS = [42, 43, 44]
MAX_EPOCHS, PATIENCE = 6, 2
BATCH, LR, WARMUP = 32, 2e-5, 0.1
WEIGHT_DECAY, MAX_LEN = 0.01, 128
FOCAL_GAMMA, POS_WEIGHT_CAP = 1.5, 5
USE_WORD_ROWS = False             # 사례집 낱말 행 제외 (Baseline 과 같다)
DEV_FRAC, DEV_SEED = 0.15, 42     # 공통 dev — 바꾸면 dev 행이 달라진다
TH_OFF = 1.01                     # 편입 대기 칸(기능성화장품_오인)의 문턱 — 꺼 둠 (D-321)
SCORE_VIEW = "v11"                # 회차 · 시드 · 문턱은 공통 dev 749행 전부로 고른다 (규약 4절) — 합성 정상 dev 로는 아무것도 고르지 않는다

# ── 🔒 정상 합성 문구를 나누는 방법 (실험 카드)
SYN_HOLD_PER_CAT, SYN_SEED = 5, 42
SYN_EXPECT = {"학습": 569, "합성 정상 dev": 285, "애매 표시": 46}      # 권소라 10-06 판 · 로컬 대조 10-08

# ── 🔒 통과 기준 (규약 5절) — Baseline 대비
GATE = {
    "prauc_drop": 0.02,           # B  : 6종 dev macro PR-AUC(시드 평균) 하락 허용폭
    "recall_drop": 0.02,          # G1 : 위반 탐지율(유형 있는 위반에 후보가 하나라도) 하락 허용폭
    "type_recall_drop": 0.05,     # G2 : 유형별 재현율 하락 허용폭 (dev 양성 30 이상인 유형만)
    "neg_flag_up": 1,             # G3 : 음성(D · L) 오판정이 늘어도 되는 행 수
    "adopt_prauc": 0.008,         # 채택 후보 ① : PR-AUC 시드 평균이 이만큼 이상 오른다
}
# ── 🔒 겨냥한 지표와 목표 (실험 카드) — 합성 정상 dev 에서 후보가 서는 비율이 Baseline 재현의 이 배수 이하
TARGET = {"syn_flag_ratio_max": 0.5}
QUIET_MARGIN = 0.5                # 판정 로직의 여유 구간 — 어느 유형이든 확률이 문턱의 절반 이상이면 통과시키지 않는다 (judge_stage1.py)
CARD8_K = 0.6                     # 카드 8 의 채택 후보 — 후보 문턱 × 0.6 (참고로만 잰다)
# ── Baseline(v10)의 dev 값 — 규약 1절 · 공통 채점 스크립트로 잰 값. 통과 기준은 이 값과 견준다
BASELINE = {
    "seed_prauc": [0.7167, 0.7081, 0.7165], "det": 491, "scored": 510, "hit": 431, "mis": 60, "miss": 19, "neg": 17, "neg_n": 86,
    "type_tp": {"질병_예방치료_표방": (25, 30), "건강기능식품_오인": (33, 42), "의약품_오인": (19, 37), "거짓_과장": (293, 315), "소비자_기만": (62, 98)},
    "ad_mis": 40, "ad_scored": 253, "ad_prauc6": 0.8243, "prauc6": 0.7167,
}

assert BASE_RUN in RUNS and all(r in RUN_DEFS for r in RUNS), "RUNS 에 Baseline 재현이 있어야 한다 — 합성 정상 dev 의 기준 수가 거기서 나온다"
RUNS = [BASE_RUN] + [r for r in RUNS if r != BASE_RUN]
if SMOKE:
    SEEDS, MAX_EPOCHS = [42], 1
NOTEBOOK = "v10-card9"
def run_tag(name):
    return f"copylane-encoder-kcbert-v10-{name}"

OUT_ROOT = "/content/drive/MyDrive/copylane_v10_card9" + ("_smoke" if SMOKE else "")
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    os.makedirs(OUT_ROOT, exist_ok=True)
except Exception as e:
    OUT_ROOT = "./copylane_v10_card9_out"
    os.makedirs(OUT_ROOT, exist_ok=True)
    print("🟡 Drive 연결 안 됨 — 결과를 런타임에만 둔다(런타임이 끊기면 사라진다):", e)
print("실험:", RUNS, "· 시드", SEEDS, "· 결과 폴더", OUT_ROOT)

## 3. 데이터 + dev 분리 — 공통 dev (Baseline 노트북과 같은 셀)

1. 묶음 = 같은 원문에서 나온 행 (`inj:T1:hf:0:0` → `hf:0:0` · 그 밖은 `id` 의 `#` 앞).
2. 원천마다 묶음의 15% 를 dev 로 (시드 42 · 묶음 이름순 → 섞기).
3. 6종마다 dev 양성이 10건보다 적으면 남은 묶음에서 채운다.
4. dev 행 = dev 묶음의 **문장 단위 · 합성 아님** 행.

같은 골든이면 **749줄 · 지문 `4ebfcb231da703b5`** 가 나와야 한다 — v11 의 `dev_ids_v11_seed42.csv` 와 같은 행 · 같은 순서다. 다르면 멈춘다.

In [ ]:
def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

SEL = ["질병_예방치료_표방", "건강기능식품_오인", "의약품_오인", "거짓_과장", "소비자_기만", "후기_체험기_기만"]   # 모델을 고르는 6종 (D-321 결정 4)
NEW_CONF = ["부당_비교광고", "비방광고"]
CONF8 = SEL + NEW_CONF                       # 확정 8종 — 문턱을 dev 에서 고른다
WAITING = ["기능성화장품_오인"]              # 편입 대기 — 칸은 두고 문턱은 끈다
LABEL_LIST = CONF8 + WAITING                 # 저장 순서 (팀장 10-06)
LABEL2ID = {l: i for i, l in enumerate(LABEL_LIST)}
N_SEL, N8, N_ALL = len(SEL), len(CONF8), len(LABEL_LIST)

def norm_text(s):
    return " ".join(s.split())

def group_key(rid):
    if rid.startswith("inj:"):               # 'inj:T1:hf:0:0' → 원본 승인 문구 'hf:0:0'
        return rid.split(":", 2)[2]
    return rid.split("#")[0]

def normalize(r):
    cond, labels = r.get("조건"), list(r.get("labels") or [])
    return {"id": r["id"], "group": group_key(r["id"]), "text": r["text"], "labels": labels,
            # 조건 M · D · L 은 라벨이 적혀 있어도 양성이 아니다 (D-296 개정 2) — target 이 학습 · 채점의 정답이다
            "target": [] if cond in ("M", "D", "L") else labels,
            "split": r["split"], "provenance": r.get("provenance"), "origin": r.get("origin"),
            "zone": r.get("구역"), "cond": cond, "unit": r.get("unit")}

def is_reason(r):
    return r["provenance"] == "ftc_decisions_body" and r["zone"] == "이유"

def is_inj(r):
    return r["origin"] == "injected" or r["id"].startswith("inj:")

def bucket(r):
    # 8종 기준 (run_judge_dist.py 와 같은 갈래)
    if r["cond"] == "M": return "M"
    if r["cond"] == "D": return "D"
    if r["cond"] == "L": return "L"
    if any(l in CONF8 for l in r["labels"]): return "scored"
    if r["labels"]: return "대기"
    return "pending" if r["cond"] in ("C", "A", "B") else "neg"

golden = load_jsonl("golden.jsonl")
all_rows = [normalize(r) for r in golden]
unknown = {l for r in all_rows for l in r["labels"]} - set(LABEL_LIST)
assert not unknown, f"🔴 예상 밖 라벨: {unknown}"
train_all = [r for r in all_rows if r["split"] == "train"]
_got = (len(all_rows), len(train_all), len(all_rows) - len(train_all))
if _got != EXPECTED_ROWS:
    raise SystemExit(f"🔴 행 수가 다르다 — 받은 {_got} · 기대 {EXPECTED_ROWS} (D-176)")
del golden, all_rows      # 🔴 여기서부터 test 행은 메모리에도 없다 — 이 노트북은 train(→ train · dev)만 본다

# ══ dev 분리 — v11 (encoder_finetune_colab_v11_kcbert_1006) 섹션 3 과 같은 코드
def dev_eligible(r):
    return r["unit"] == "문장" and r["origin"] != "injected"

groups = defaultdict(list)
for r in train_all:
    groups[r["group"]].append(r)

prov_groups = defaultdict(list)             # 원천별 15% (묶음 단위 · 시드 고정)
for g in sorted(groups):
    elig = [r for r in groups[g] if dev_eligible(r)]
    if elig:
        prov_groups[Counter(r["provenance"] for r in elig).most_common(1)[0][0]].append(g)
rng = random.Random(DEV_SEED)
dev_groups = set()
for prov in sorted(prov_groups):
    gs = prov_groups[prov][:]
    rng.shuffle(gs)
    dev_groups.update(gs[:max(1, math.ceil(DEV_FRAC * len(gs)))])

def conf_pos(rows, l):
    return sum(1 for r in rows if dev_eligible(r) and l in r["target"])
rest = [g for g in sorted(groups) if g not in dev_groups]
rng.shuffle(rest)
for l in SEL:                               # 6종마다 dev 양성이 10건보다 적으면 채운다
    total = conf_pos(train_all, l)
    want = min(10, math.ceil(DEV_FRAC * total))
    have = sum(conf_pos(groups[g], l) for g in dev_groups)
    for g in rest:
        if have >= want:
            break
        k = conf_pos(groups[g], l)
        if k and g not in dev_groups:
            dev_groups.add(g); have += k

dev_rows = [r for g in sorted(dev_groups) for r in groups[g] if dev_eligible(r)]
dev_texts = {norm_text(r["text"]) for r in dev_rows}
DEV_SHA = hashlib.sha256("|".join(sorted(r["id"] for r in dev_rows)).encode()).hexdigest()[:16]
DEV_EXPECT = {"529c970556d77bd2c231526e20531294bf469f794e74a4796835a36f456eb7fe": (749, "4ebfcb231da703b5")}   # 골든 sha → v11 dev (줄 수 · 지문)
if GOLDEN_SHA in DEV_EXPECT:
    if (len(dev_rows), DEV_SHA) != DEV_EXPECT[GOLDEN_SHA]:
        raise SystemExit(f"🔴 dev 가 v11 과 다르다 — {len(dev_rows)}줄 · 지문 {DEV_SHA} · 기대 {DEV_EXPECT[GOLDEN_SHA]}. 분리 코드를 건드렸는지 본다")
    print(f"✅ dev = v11 과 같은 행: 묶음 {len(dev_groups)}개 · {len(dev_rows)}줄 · 지문 {DEV_SHA}")
else:
    print(f"🟡 이 골든의 v11 dev 지문을 모른다 — 대조 없이 간다: 묶음 {len(dev_groups)}개 · {len(dev_rows)}줄 · 지문 {DEV_SHA}")
assert len({r["id"] for r in dev_rows}) == len(dev_rows), "dev 에 같은 id 가 둘 이상이다"

# ── dev 행의 갈래 · 채점 방식
for r in dev_rows:
    r["bucket"] = bucket(r)
B = np.array([r["bucket"] for r in dev_rows])
IS_REASON = np.array([is_reason(r) for r in dev_rows])
VIEW = {"v11": np.ones(len(dev_rows), dtype=bool),
        "A": ~IS_REASON & ~np.isin(B, ["M", "pending"])}
SEL_MASK = VIEW[SCORE_VIEW]
BUCKETS = ["scored", "대기", "pending", "M", "D", "L", "neg"]
BUCKET_KO = {"scored": "유형 있는 위반", "대기": "편입 대기 유형만", "pending": "유형 없는 위반", "M": "조건 M", "D": "조건 D", "L": "조건 L(적법)", "neg": "그 밖 음성"}
print("\ndev 갈래 (전체 / 그중 「이유」 구역):")
for b in BUCKETS:
    if (B == b).any():
        print(f"  {BUCKET_KO[b]:12s} {int((B == b).sum()):4d} / {int(((B == b) & IS_REASON).sum()):4d}")
print(f"  합계           {len(dev_rows):4d} / {int(IS_REASON.sum()):4d}")
print(f"채점 행 — v11 방식 {int(VIEW['v11'].sum())}줄 · A 방식 {int(VIEW['A'].sum())}줄  → 이번 선택 방식: {SCORE_VIEW}")

## 4. 학습셋 · 정상 합성 문구 나누기

**골든에서 오는 학습 행은 Baseline 과 같다** (3,964행). dev 묶음을 뺀 train 에서 순서대로:
dev 와 같은 문구 제거 → 조건 M 제외 → 유형 없는 위반 제외 → 낱말 행 제외 → 같은 문구 병합.

정상 합성 문구 900건은 이렇게 나눈다.

| 갈래 | 행 | 쓰임 |
|---|---:|---|
| 학습 | 569 | 카드 9 의 학습에 **유형 없는 음성**으로 더한다 (제품 30종 · 애매 표시 제외) |
| 합성 정상 dev | 285 | 학습에 넣지 않는다. 목표를 여기서 잰다 (품목마다 제품 5종 · 애매 표시 제외) |
| 애매 표시 | 46 | 학습에도 목표에도 넣지 않는다. 참고로만 잰다 |

- **제품 단위로 뗀다** — 같은 제품의 문구가 학습과 측정에 함께 들어가지 않는다. 다만 생성기 · 각도 · 문체는 같다(한계).
- 골든의 학습 행 · 공통 dev 와 같은 문구가 있으면 멈춘다.
- 🔴 합성 정상 dev 로는 **회차 · 시드 · 문턱을 고르지 않는다.** 그것은 공통 dev 가 한다.

In [ ]:
base_pool = [r for r in train_all if r["group"] not in dev_groups]
STEPS = [("dev 묶음 뺀 train", len(base_pool))]
pool0 = [r for r in base_pool if norm_text(r["text"]) not in dev_texts];                 STEPS.append(("dev 와 같은 문구 제거", len(pool0)))
pool0 = [r for r in pool0 if r["cond"] != "M"];                                          STEPS.append(("조건 M 제외", len(pool0)))
pool0 = [r for r in pool0 if not (r["cond"] in ("C", "A", "B") and not r["labels"])];    STEPS.append(("유형 없는 위반 제외", len(pool0)))
if not USE_WORD_ROWS:
    pool0 = [r for r in pool0 if r["unit"] != "낱말"];                                   STEPS.append(("낱말 행 제외", len(pool0)))
print(" → ".join(f"{k} {n:,}" for k, n in STEPS))
assert not any(r["cond"] == "M" for r in pool0) and not any(r["target"] for r in pool0 if r["cond"] in ("D", "L"))

merged = {}
for r in pool0:                                      # 같은 문구 병합 — 라벨 합집합 · 이유/비이유는 따로 (Baseline 과 같은 방식)
    k = (norm_text(r["text"]), is_reason(r))
    if k in merged:
        merged[k]["target"] = sorted(set(merged[k]["target"]) | set(r["target"]))
    else:
        merged[k] = dict(r, target=list(r["target"]))
BASE_TRAIN = list(merged.values())

# ── 정상 합성 문구 — 제품 단위로 나눈다
with open(SYN_NAME, encoding="utf-8-sig", newline="") as f:
    _raw = list(csv.DictReader(f))
assert len(_raw) == 900 and len({r["id"] for r in _raw}) == 900, "정상 합성 문구가 900행이 아니다"
SYN_CATS = ["식품", "건기식", "화장품"]
def syn_row(r):
    return {"id": "syn:" + r["id"], "group": f"syn:{r['품목']}:{r['제품종류']}", "text": r["text"], "labels": [], "target": [],
            "split": "extra", "provenance": "llm_copy_ksr_20261006", "origin": "synthetic_lawful", "zone": None, "cond": "정상합성", "unit": "문장",
            "cat": r["품목"], "product": r["제품종류"], "angle": r["각도"], "style": r["문체"], "amb": bool(r["애매표시"].strip())}
SYN = [syn_row(r) for r in _raw]
assert {r["cat"] for r in SYN} == set(SYN_CATS)
SYN_HOLD = set()
for c in SYN_CATS:
    prods = sorted({r["product"] for r in SYN if r["cat"] == c})
    SYN_HOLD |= {(c, p) for p in random.Random(SYN_SEED).sample(prods, SYN_HOLD_PER_CAT)}
SYN_AMB = [r for r in SYN if r["amb"]]
SYN_TRAIN = [r for r in SYN if not r["amb"] and (r["cat"], r["product"]) not in SYN_HOLD]
SYN_DEV = [r for r in SYN if not r["amb"] and (r["cat"], r["product"]) in SYN_HOLD]
SYN_EVAL = SYN_DEV + SYN_AMB                          # 학습에 넣지 않는 행 — 시드마다 확률을 적어 둔다
_got = {"학습": len(SYN_TRAIN), "합성 정상 dev": len(SYN_DEV), "애매 표시": len(SYN_AMB)}
if _got != SYN_EXPECT:
    raise SystemExit(f"🔴 정상 합성 문구를 나눈 수가 카드와 다르다 — 받은 {_got} · 기대 {SYN_EXPECT}")
_base_texts = {norm_text(r["text"]) for r in BASE_TRAIN}
_dup = [r["id"] for r in SYN if norm_text(r["text"]) in dev_texts or norm_text(r["text"]) in _base_texts]
if _dup:
    raise SystemExit(f"🔴 정상 합성 문구 {len(_dup)}행이 골든의 학습 행 · 공통 dev 와 같은 문구다 (예: {_dup[:3]}) — 음성으로 넣으면 라벨이 부딪친다")
assert not ({r["group"] for r in SYN_TRAIN} & {r["group"] for r in SYN_EVAL if not r["amb"]}), "같은 제품이 학습과 합성 정상 dev 에 함께 있다"
assert not ({norm_text(r["text"]) for r in SYN_TRAIN} & {norm_text(r["text"]) for r in SYN_EVAL}), "학습과 측정에 같은 문구가 있다"
print("\n■ 정상 합성 문구 나누기 (품목별)")
display(pd.DataFrame({k: {c: sum(r["cat"] == c for r in rs) for c in SYN_CATS} | {"합계": len(rs)}
                      for k, rs in (("학습", SYN_TRAIN), ("합성 정상 dev", SYN_DEV), ("애매 표시", SYN_AMB))}).T)
print("측정용으로 뗀 제품:", {c: sorted(p for cc, p in SYN_HOLD if cc == c) for c in SYN_CATS})

TRAIN, TRAIN_STATS = {}, {}
for name, cfg in RUN_DEFS.items():
    rows = list(BASE_TRAIN) + (list(SYN_TRAIN) if cfg["syn"] else [])
    if SMOKE:
        rows = list(BASE_TRAIN)
        random.Random(0).shuffle(rows)
        rows = rows[:300] + (SYN_TRAIN[:60] if cfg["syn"] else [])
    TRAIN[name] = rows
    TRAIN_STATS[name] = {"학습 행": len(rows), "합성(주입)": sum(map(is_inj, rows)), "이유 구역": sum(map(is_reason, rows)),
                         "정상 합성(음성)": sum(r["origin"] == "synthetic_lawful" for r in rows), "음성": sum(1 for r in rows if not r["target"]),
                         **{l: sum(l in r["target"] for r in rows) for l in LABEL_LIST}}
print("\n■ 실험별 학습셋 (행 수 · 유형별 양성)" + ("   (SMOKE — 줄였다)" if SMOKE else ""))
display(pd.DataFrame(TRAIN_STATS).T)
EXPECT_TRAIN = {"Baseline재현": 3964, "카드9-정상음성": 4533}       # 재동결 10-05 판 · 로컬 대조 10-08
if GOLDEN_SHA in DEV_EXPECT and not SMOKE and USE_WORD_ROWS is False:
    _got = {n: TRAIN_STATS[n]["학습 행"] for n in RUN_DEFS}
    if _got != EXPECT_TRAIN:
        raise SystemExit(f"🔴 학습 행 수가 기대와 다르다 — 받은 {_got} · 기대 {EXPECT_TRAIN}. 학습셋 코드를 건드렸는지 본다")
    print("✅ 학습 3,964행 (Baseline 과 같다) · 카드 9 는 정상 합성 569행을 더한 4,533행")
for name in RUNS:
    assert not (set(r["id"] for r in TRAIN[name]) & (set(r["id"] for r in dev_rows) | set(r["id"] for r in SYN_EVAL))), "학습 행에 dev · 측정 행이 섞였다"
    assert not ({norm_text(r["text"]) for r in TRAIN[name]} & dev_texts), "학습 행에 dev 와 같은 문구가 있다"
    assert all(r["split"] == "train" or r["origin"] == "synthetic_lawful" for r in TRAIN[name]), "골든 train 도 정상 합성도 아닌 행이 학습에 있다"

## 5. 손실 · 지표 — Baseline 과 같은 셀

- Focal Loss(γ 1.5) · α = pos_weight = √(음성/양성) 을 1 ~ 5 로 자른 값. **실험마다 그 실험의 학습 행으로 계산한다**(v10 노트북과 같은 방식) — 음성이 늘면 조금 커진다.
- 회차(조기 종료) · 시드 = 공통 dev 의 **6종 macro PR-AUC** 가 가장 높은 것. 문턱 = 확정 8종마다 공통 dev F1 최대 (규약 4절).

In [ ]:
import torch
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments,
                          EarlyStoppingCallback, DataCollatorWithPadding, set_seed)
from sklearn.metrics import average_precision_score, f1_score

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
collator = DataCollatorWithPadding(tokenizer)

def multihot(labels):
    v = np.zeros(N_ALL, dtype=np.float32)
    for l in labels:
        v[LABEL2ID[l]] = 1.0
    return v

def to_ds(rows):
    ds = Dataset.from_dict({"text": [r["text"] for r in rows], "labels": [multihot(r["target"]) for r in rows]})
    ds = ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=MAX_LEN), batched=True)
    ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
    return ds

Y_DEV = np.stack([multihot(r["target"]) for r in dev_rows]).astype(int)
dev_ds = to_ds(dev_rows)

def pos_weight_of(rows):
    Y = np.stack([multihot(r["target"]) for r in rows])
    pos = Y.sum(0); neg = len(Y) - pos
    return np.clip(np.sqrt(neg / np.clip(pos, 1e-6, None)), 1, POS_WEIGHT_CAP)

class FocalTrainer(Trainer):
    def __init__(self, *args, pos_weight=None, **kwargs):
        super().__init__(*args, **kwargs)
        self._pw = torch.tensor(pos_weight, dtype=torch.float32)
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels").float()
        outputs = model(**inputs)
        logits = outputs.logits.float()
        bce = torch.nn.functional.binary_cross_entropy_with_logits(logits, labels, reduction="none")
        p = torch.sigmoid(logits)
        p_t = p * labels + (1 - p) * (1 - labels)
        loss = ((self._pw.to(logits.device) * labels + (1 - labels)) * (1 - p_t) ** FOCAL_GAMMA * bce).mean()
        return (loss, outputs) if return_outputs else loss

def safe(fn, y, p):
    return float(fn(y, p)) if 0 < y.sum() < len(y) else None

def macro_prauc(y, p, n=N_SEL):
    # 앞 n 칸의 PR-AUC 평균 — 양성이 있는 유형만. n=6 이 모델을 고르는 기준이다
    v = [safe(average_precision_score, y[:, i], p[:, i]) for i in range(n)]
    v = [x for x in v if x is not None]
    return float(np.mean(v)) if v else float("nan")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    p = 1 / (1 + np.exp(-logits))
    return {"macro_prauc": macro_prauc(labels[SEL_MASK].astype(int), p[SEL_MASK])}

TH_GRID = np.arange(0.1, 0.9001, 0.025)
def pick_thresholds(y, p):
    th = {}
    for i, l in enumerate(CONF8):
        if y[:, i].sum() < 10:
            th[l] = 0.5
            continue
        fs = [f1_score(y[:, i], p[:, i] >= t, zero_division=0) for t in TH_GRID]
        th[l] = float(round(TH_GRID[int(np.argmax(fs))], 3))
    th[WAITING[0]] = TH_OFF
    return th

def predict_probs(mdl, rows, bs=64):
    mdl.eval()
    dev_ = next(mdl.parameters()).device
    out = []
    for s in range(0, len(rows), bs):
        enc = tokenizer([r["text"] for r in rows[s:s + bs]], truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt").to(dev_)
        with torch.no_grad():
            out.append(torch.sigmoid(mdl(**enc).logits.float()).cpu().numpy())
    return np.concatenate(out)

print("dev 양성 (선택 방식", SCORE_VIEW, "):", {l: int(Y_DEV[SEL_MASK, i].sum()) for i, l in enumerate(LABEL_LIST)})
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 — CPU 로는 매우 느리다. 런타임 유형을 GPU 로 바꾼다")

## 6. 학습 — 실험 × 시드

🔒 **기준과 목표를 먼저 찍고 학습한다.** 시드 하나가 끝나면 공통 dev 확률과 합성 정상 문구(측정용 285 + 애매 46)의 확률을 Drive 에 적는다 — 다시 실행하면 끝난 시드는 건너뛴다(`♻️`).

In [ ]:
print("🔒 통과 기준 (규약 5절) — Baseline(v10 · 규약 값) 대비 · 가장 좋은 시드 · 그 시드의 dev 문턱")
print(f"  B   6종 dev macro PR-AUC(시드 평균) 하락이 {GATE['prauc_drop']} 이내   (Baseline {np.mean(BASELINE['seed_prauc']):.4f})")
print(f"  G1  위반 탐지율 하락이 {GATE['recall_drop'] * 100:.0f}%p 이내   (Baseline {BASELINE['det']}/{BASELINE['scored']})")
print(f"  G2  유형별 재현율 하락이 {GATE['type_recall_drop'] * 100:.0f}%p 이내 (dev 양성 30 이상인 유형만)")
print(f"  G3  음성(D · L) 오판정이 +{GATE['neg_flag_up']}행 이내   (Baseline {BASELINE['neg']}/{BASELINE['neg_n']})")
print("🔒 겨냥한 지표와 목표 (실험 카드)")
print(f"  카드 9  합성 정상 dev {len(SYN_DEV)}행 중 후보가 서는 비율이 Baseline 재현의 {TARGET['syn_flag_ratio_max']:g}배 이하 (가장 좋은 시드 · 그 시드의 문턱)")
print(f"  채택 후보 = 통과 기준에 걸리지 않고 + 목표를 채운다")
print("  ※ 통과 기준의 판정은 공통 채점 스크립트(score_encoder_dev.py)로 한다. 아래 표는 미리 보는 값이다.\n")

CFG_SIG = [NOTEBOOK, GOLDEN_SHA, DEV_SHA, SCORE_VIEW, MODEL_NAME, MAX_EPOCHS, PATIENCE, BATCH, LR, WARMUP, WEIGHT_DECAY, MAX_LEN,
           FOCAL_GAMMA, POS_WEIGHT_CAP, USE_WORD_ROWS, SMOKE]
def sig_of(name, seed):
    ids = hashlib.sha256("|".join(r["id"] for r in TRAIN[name]).encode()).hexdigest()
    ev = hashlib.sha256("|".join(r["id"] for r in SYN_EVAL).encode()).hexdigest()
    return hashlib.sha256(json.dumps(CFG_SIG + [name, RUN_DEFS[name], seed, ids, SYN_SHA, ev], ensure_ascii=False).encode()).hexdigest()[:16]

def train_seed(name, seed, get_ds, pw):
    tag = run_tag(name)
    keep, local = f"{OUT_ROOT}/{tag}/seed{seed}", f"./runs/{tag}/seed{seed}"
    done = f"{keep}/done.json"
    if os.path.exists(done) and os.path.exists(f"{keep}/dev_p.npy") and os.path.exists(f"{keep}/syn_p.npy"):
        res = json.load(open(done, encoding="utf-8"))
        if res.get("sig") == sig_of(name, seed):
            res["dev_p"] = np.load(f"{keep}/dev_p.npy")
            res["syn_p"] = np.load(f"{keep}/syn_p.npy")
            print(f"♻️ {name} · seed {seed}: 이미 끝남 — 건너뜀 (best epoch {res['best_epoch']})")
            return res
        print(f"{name} · seed {seed}: 저장된 기록이 지금 설정과 다르다 — 다시 학습한다")
    t0 = time.time()
    train_ds = get_ds()
    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=N_ALL, problem_type="multi_label_classification",
        id2label=dict(enumerate(LABEL_LIST)), label2id=LABEL2ID)
    args = TrainingArguments(
        output_dir=f"./runs/{tag}/ckpt{seed}", eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
        per_device_train_batch_size=BATCH, per_device_eval_batch_size=64, num_train_epochs=MAX_EPOCHS,
        learning_rate=LR, weight_decay=WEIGHT_DECAY, lr_scheduler_type="linear",
        warmup_steps=int(WARMUP * math.ceil(len(train_ds) / BATCH) * MAX_EPOCHS),
        load_best_model_at_end=True, metric_for_best_model="macro_prauc", greater_is_better=True,
        logging_steps=50, report_to="none", remove_unused_columns=False, seed=seed,
        fp16=torch.cuda.is_available())
    trainer = FocalTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=dev_ds, data_collator=collator,
                           compute_metrics=compute_metrics, pos_weight=pw,
                           callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)])
    trainer.train()
    hist = [h for h in trainer.state.log_history if "eval_macro_prauc" in h]
    best = max(hist, key=lambda h: h["eval_macro_prauc"])
    dev_p = predict_probs(trainer.model, dev_rows)
    syn_p = predict_probs(trainer.model, SYN_EVAL)          # 🆕 합성 정상 문구(측정용 + 애매) — 학습에 넣지 않은 행이다
    shutil.rmtree(local, ignore_errors=True)
    trainer.save_model(local)
    shutil.rmtree(f"./runs/{tag}/ckpt{seed}", ignore_errors=True)
    res = {"sig": sig_of(name, seed), "seed": seed, "best_epoch": int(round(best["epoch"])), "epochs_run": int(round(hist[-1]["epoch"])),
           "curve": [[int(round(h["epoch"])), round(h["eval_macro_prauc"], 4)] for h in hist], "minutes": round((time.time() - t0) / 60, 1)}
    os.makedirs(keep, exist_ok=True)
    np.save(f"{keep}/dev_p.npy", dev_p)
    np.save(f"{keep}/syn_p.npy", syn_p)
    shutil.copy(f"{local}/config.json", f"{keep}/config.json")
    json.dump(res, open(done, "w", encoding="utf-8"), ensure_ascii=False)       # 🔴 done.json 을 맨 끝에 쓴다 — 있으면 이 시드는 끝난 것이다
    res["dev_p"], res["syn_p"] = dev_p, syn_p
    del trainer, model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return res

def scheme_of(name, R):
    s, st = R["best_seed"], TRAIN_STATS[name]
    return {
        "label_list": LABEL_LIST, "label_thresholds": R["thresholds"],
        "confirmed_labels": CONF8, "waiting_labels": WAITING, "legacy_confirmed_labels": SEL,
        "waiting_note": f"편입 대기(D-321) — 칸은 두되 문턱 {TH_OFF} 로 꺼 둔다",
        "selection_note": "회차 · 시드는 6종 dev macro PR-AUC 로 고른다 (D-321 결정 4)",
        "backbone": MODEL_NAME, "notebook": f"{NOTEBOOK} (박수진 · 실험 카드 9 — 정상 합성 문구를 음성으로)", "experiment": run_tag(name),
        "baseline": "copylane-encoder-kcbert-v10-합성O-이유O", "use_synthetic_lawful": RUN_DEFS[name]["syn"],
        "synthetic_lawful_rows": st["정상 합성(음성)"], "synthetic_lawful_sha256": SYN_SHA,
        "synthetic_lawful_note": "권소라 카피 생성기의 정상 문구(사람 판독 전) — 제품 단위로 나눠 일부만 학습 음성으로 썼다. 애매 표시 제외",
        "use_injected": True, "use_reason_zone": True, "use_word_rows": USE_WORD_ROWS,
        "seed": s, "best_epoch": R["seeds"][s]["best_epoch"], "golden_sha256": GOLDEN_SHA, "golden_rows": list(EXPECTED_ROWS),
        "train_rows": st["학습 행"], "train_injected_rows": st["합성(주입)"], "train_reason_rows": st["이유 구역"],
        "train_label_pos": {l: st[l] for l in LABEL_LIST},
        "train_rule": "조건 M 제외 · D · L 음성 · 유형 없는 위반 제외" + ("" if USE_WORD_ROWS else " · 낱말 행 제외")
                      + (" · 정상 합성 문구를 음성으로 더함" if RUN_DEFS[name]["syn"] else ""),
        "dev_rule": f"v11 과 같다 — 원천별 묶음 {DEV_FRAC:.0%} + 6종 보충 · 시드 {DEV_SEED} · 문장 단위 · 합성 아님",
        "dev_sha": DEV_SHA, "dev_rows": len(dev_rows), "score_view": SCORE_VIEW, "score_rows": int(SEL_MASK.sum()),
        "score_view_note": "v11 = dev 전부(M · 유형 없는 위반 · D · L 음성) / A = 이유 구역 · M · 유형 없는 위반을 뺀 줄",
        "threshold_note": "가장 좋은 시드의 dev F1 최대 · 격자 0.1~0.9(0.025) · dev 양성 10건 미만이면 0.5. 적용은 확률 >= threshold.",
        "thresholds_by_seed": {str(k): v["thresholds"] for k, v in R["seeds"].items()},
        "dev_macro_prauc6_by_seed": {str(k): v["prauc6"] for k, v in R["seeds"].items()},
        "final_test": False, "test_note": "이 노트북은 test 를 채점하지 않는다 (D-175)", "smoke": SMOKE,
    }

def export_run(name, R):
    """시드별 dev 행별 확률 CSV(규약 3절) · 시드별 합성 정상 행별 확률 CSV · 문턱 zip · (스위치) 가장 좋은 시드의 모델 zip 을 결과 폴더에 둔다."""
    tag, s = run_tag(name), R["best_seed"]
    scheme, out = scheme_of(name, R), []
    for sd in SEEDS:
        csv_path = f"{OUT_ROOT}/{tag}_seed{sd}_dev행별확률.csv"
        with open(csv_path, "w", encoding="utf-8-sig", newline="") as f:      # 문구는 넣지 않는다 (id · 갈래 · 확률만)
            w = csv.writer(f)
            w.writerow(["id", "조건", "bucket", "이유구역"] + [f"p_{l}" for l in LABEL_LIST])
            for r, row in zip(dev_rows, R["seeds"][sd]["dev_p"]):
                w.writerow([r["id"], r["cond"] or "", r["bucket"], int(is_reason(r))] + [f"{x:.6f}" for x in row])
        out.append(os.path.basename(csv_path))
        csv_path = f"{OUT_ROOT}/{tag}_seed{sd}_합성정상_행별확률.csv"
        with open(csv_path, "w", encoding="utf-8-sig", newline="") as f:      # 문구는 넣지 않는다 (id · 품목 · 갈래 · 확률만)
            w = csv.writer(f)
            w.writerow(["id", "품목", "갈래"] + [f"p_{l}" for l in LABEL_LIST])
            for r, row in zip(SYN_EVAL, R["seeds"][sd]["syn_p"]):
                w.writerow([r["id"], r["cat"], "애매" if r["amb"] else "합성정상dev"] + [f"{x:.6f}" for x in row])
        out.append(os.path.basename(csv_path))
    with zipfile.ZipFile(f"{OUT_ROOT}/{tag}_scheme.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.writestr("label_scheme.json", json.dumps(scheme, ensure_ascii=False, indent=2))
        z.write(f"{OUT_ROOT}/{tag}/seed{s}/config.json", "config.json")
    out.append(f"{tag}_scheme.zip")
    local = f"./runs/{tag}/seed{s}"
    if not SAVE_MODEL_ZIP or name == BASE_RUN:           # Baseline 재현의 모델은 v10 과 같은 것이라 저장하지 않는다
        pass
    elif not os.path.exists(f"{local}/config.json"):
        if os.path.exists(f"{OUT_ROOT}/{tag}.zip") and R.get("zip_seed") == s:
            out.append(f"{tag}.zip (앞서 저장한 것)")
        else:
            print(f"  🟡 {name}: 가장 좋은 시드 {s} 의 가중치가 이 런타임에 없다(앞 세션에서 학습) — 모델 zip 은 못 만든다. "
                  f"필요하면 Drive 의 {tag} 폴더를 지우고 이 실험만 다시 돌린다")
    else:
        sdir = f"./export/{tag}"
        shutil.rmtree(sdir, ignore_errors=True)
        m = AutoModelForSequenceClassification.from_pretrained(local)
        assert [m.config.id2label[i] for i in range(m.config.num_labels)] == LABEL_LIST, "저장 칸 순서가 다르다 (D-220)"
        m.save_pretrained(sdir, safe_serialization=True)
        tokenizer.save_pretrained(sdir)
        assert os.path.exists(f"{sdir}/tokenizer.json"), "tokenizer.json 이 없다 (앱 로더 필수 파일)"
        json.dump(scheme, open(f"{sdir}/label_scheme.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
        try:
            zp = shutil.make_archive(f"./export/{tag}", "zip", sdir)
            shutil.copy(zp, f"{OUT_ROOT}/{tag}.zip")
            json.dump({"zip_seed": s}, open(f"{OUT_ROOT}/{tag}/zip_seed.json", "w"))
            out.append(f"{tag}.zip")
            os.remove(zp); shutil.rmtree(sdir, ignore_errors=True)          # 런타임 디스크 — 결과 폴더에 옮겼으면 지운다
        except Exception as e:
            print(f"  🟡 {name}: 모델 zip 저장 안 됨 (Drive 용량?) — {e}")
        del m
    return out

RESULTS = {}
T_ALL = time.time()
for name in RUNS:
    tag, rows = run_tag(name), TRAIN[name]
    pw = pos_weight_of(rows)
    print(f"\n{'=' * 22} {name} · 학습 {len(rows):,}행 {'=' * 22}")
    print("pos_weight:", {l: round(float(pw[i]), 2) for i, l in enumerate(LABEL_LIST)})
    _cache, seeds = {}, {}
    def get_ds():                                 # 학습할 시드가 있을 때만 토크나이즈한다
        if "ds" not in _cache:
            _cache["ds"] = to_ds(rows)
        return _cache["ds"]
    for seed in SEEDS:
        res = train_seed(name, seed, get_ds, pw)
        y, p = Y_DEV[SEL_MASK], res["dev_p"][SEL_MASK]
        res["prauc6"], res["prauc8"] = macro_prauc(y, p, N_SEL), macro_prauc(y, p, N8)
        res["thresholds"] = pick_thresholds(y, p)
        seeds[seed] = res
        print(f"  seed {seed}: 6종 dev macro PR-AUC {res['prauc6']:.4f} (8종 {res['prauc8']:.4f}) · best epoch {res['best_epoch']}/{res['epochs_run']} · 곡선 {res['curve']} · {res['minutes']}분")
    best = max(SEEDS, key=lambda s: seeds[s]["prauc6"])
    _zs = f"{OUT_ROOT}/{tag}/zip_seed.json"
    R = {"seeds": seeds, "best_seed": best, "thresholds": seeds[best]["thresholds"],
         "prauc6_mean": float(np.mean([seeds[s]["prauc6"] for s in SEEDS])), "prauc6_std": float(np.std([seeds[s]["prauc6"] for s in SEEDS])),
         "zip_seed": json.load(open(_zs))["zip_seed"] if os.path.exists(_zs) else None}
    RESULTS[name] = R
    print(f"[{name}] 6종 dev macro PR-AUC {R['prauc6_mean']:.4f} ± {R['prauc6_std']:.4f} · 가장 좋은 시드 {best}")
    print("  문턱:", {l: R["thresholds"][l] for l in CONF8})
    print("  저장:", export_run(name, R))
    for s in SEEDS:                               # 디스크 — 가장 좋은 시드의 가중치만 남긴다
        if s != best:
            shutil.rmtree(f"./runs/{tag}/seed{s}", ignore_errors=True)
print(f"\n학습 끝 — {round((time.time() - T_ALL) / 60, 1)}분")

## 7. 결과 — 미리 보기

- **표 1** 공통 dev — 가장 좋은 시드 · 그 시드의 문턱으로 잰 값. 맨 윗줄은 규약의 Baseline(v10) 값이다.
- **표 2** 유형별 재현율 · 정밀도.
- **표 3** 통과 기준(B · G1 · G2 · G3) — 규약의 Baseline 값 대비.
- **표 4** 🆕 합성 정상 문구 — 후보가 서는 비율 · 조용한 비율(판정 로직에서 통과할 수 있는 문장) · 품목별 · 유형별. **목표는 여기서 잰다.**

읽는 법
- 「후보가 선다」 = 확정 8종 중 하나라도 확률이 문턱 이상. 「조용하다」 = 모든 유형의 확률이 문턱의 절반 미만 — 판정 로직은 이 문장만 통과시킨다(사전이 조용할 때).
- 「카드 8 문턱」 = 후보 문턱을 0.6배로 낮춘 자리(카드 8 의 채택 후보). 참고로만 본다.
- Baseline 재현이 v10 과 같은 수를 내는지는 표 3 아래 한 줄로 확인한다.

In [ ]:
SC = B == "scored"
NEG = np.isin(B, ["D", "L", "neg"])
AD = ~IS_REASON & ~np.isin(B, ["M", "pending"])          # 광고 문구 줄
Y8 = Y_DEV[:, :N8] == 1

def measure(p, th):
    fl = p[:, :N8] >= np.array([th[l] for l in CONF8])
    any_, hit = fl.any(1), (fl & Y8).any(1)
    mis = any_ & ~hit
    out = {"scored": int(SC.sum()), "det": int(any_[SC].sum()), "hit": int(hit[SC].sum()), "mis": int(mis[SC].sum()),
           "miss": int((~any_)[SC].sum()), "neg": int(any_[NEG].sum()), "neg_n": int(NEG.sum()),
           "D": (int(any_[B == "D"].sum()), int((B == "D").sum())), "L": (int(any_[B == "L"].sum()), int((B == "L").sum())),
           "ad_scored": int((SC & AD).sum()), "ad_det": int((any_ & SC & AD).sum()), "ad_mis": int((mis & SC & AD).sum()),
           "prauc6": macro_prauc(Y_DEV, p), "prauc8": macro_prauc(Y_DEV, p, N8), "ad_prauc6": macro_prauc(Y_DEV[AD], p[AD]),
           "cand_avg": float(fl[SC & any_].sum(1).mean()) if (SC & any_).any() else float("nan")}
    out["type"] = {l: {"n": int(Y8[:, i].sum()), "tp": int((fl[:, i] & Y8[:, i]).sum()), "pp": int(fl[:, i].sum())} for i, l in enumerate(CONF8)}
    return out

SYN_IS_DEV = np.array([not r["amb"] for r in SYN_EVAL])
SYN_CAT = np.array([r["cat"] for r in SYN_EVAL])
def syn_measure(p, th):
    """합성 정상 문구 — 후보가 선 행 · 조용한 행 · 카드 8 문턱에서 후보가 선 행 (측정용 285행 / 애매 46행 따로)."""
    ratio = p[:, :N8] / np.array([th[l] for l in CONF8])
    flag, quiet, flag8 = (ratio >= 1).any(1), ratio.max(1) < QUIET_MARGIN, (ratio >= CARD8_K).any(1)
    out = {}
    for key, m in (("dev", SYN_IS_DEV), ("amb", ~SYN_IS_DEV)):
        out[key] = {"n": int(m.sum()), "flag": int(flag[m].sum()), "quiet": int(quiet[m].sum()), "flag_card8": int(flag8[m].sum()),
                    "by_cat": {c: (int(flag[m & (SYN_CAT == c)].sum()), int(quiet[m & (SYN_CAT == c)].sum()), int((m & (SYN_CAT == c)).sum())) for c in SYN_CATS},
                    "by_type": {l: int((ratio[m, i] >= 1).sum()) for i, l in enumerate(CONF8)},
                    "max_ratio_median": float(np.median(ratio[m].max(1))) if m.any() else float("nan")}
    return out

def rate(k, n):
    return "—" if n == 0 else f"{k}/{n} ({k / n * 100:.1f}%)"

M, SYN_M = {}, {}
for name in RUNS:
    R = RESULTS[name]
    M[name] = measure(R["seeds"][R["best_seed"]]["dev_p"], R["thresholds"])
    M[name]["per_seed"] = {s: measure(R["seeds"][s]["dev_p"], R["seeds"][s]["thresholds"]) for s in SEEDS}
    SYN_M[name] = syn_measure(R["seeds"][R["best_seed"]]["syn_p"], R["thresholds"])
    SYN_M[name]["per_seed"] = {s: syn_measure(R["seeds"][s]["syn_p"], R["seeds"][s]["thresholds"]) for s in SEEDS}
b = BASELINE
b_mean, b_std = float(np.mean(b["seed_prauc"])), float(np.std(b["seed_prauc"]))

# ── 표 1
_t = {"Baseline v10 (규약 값)": {
    "6종 PR-AUC 시드 평균": f"{b_mean:.4f} ± {b_std:.4f}", "선택 시드": "42", "선택 시드 PR-AUC (dev 전체 · 광고 문구 줄)": f"{b['prauc6']:.3f} · {b['ad_prauc6']:.3f}",
    "위반 탐지 ↑": rate(b["det"], b["scored"]), "정답 유형 적중 ↑": rate(b["hit"], b["scored"]), "유형 오인 ↓": rate(b["mis"], b["scored"]),
    "놓침 ↓": rate(b["miss"], b["scored"]), "음성(D · L) 오판정 ↓": f"{b['neg']}/{b['neg_n']}", "광고 문구 줄의 위반 탐지 ↑": "—"}}
for name in RUNS:
    R, m = RESULTS[name], M[name]
    _t[name] = {
        "6종 PR-AUC 시드 평균": f"{R['prauc6_mean']:.4f} ± {R['prauc6_std']:.4f}", "선택 시드": f"{R['best_seed']} (epoch {R['seeds'][R['best_seed']]['best_epoch']})",
        "선택 시드 PR-AUC (dev 전체 · 광고 문구 줄)": f"{m['prauc6']:.3f} · {m['ad_prauc6']:.3f}",
        "위반 탐지 ↑": rate(m["det"], m["scored"]), "정답 유형 적중 ↑": rate(m["hit"], m["scored"]), "유형 오인 ↓": rate(m["mis"], m["scored"]),
        "놓침 ↓": rate(m["miss"], m["scored"]), "음성(D · L) 오판정 ↓": f"{m['neg']}/{m['neg_n']} (D {m['D'][0]} · L {m['L'][0]})",
        "광고 문구 줄의 위반 탐지 ↑": rate(m["ad_det"], m["ad_scored"])}
print(f"■ 표 1 · 공통 dev {len(dev_rows)}행 — 가장 좋은 시드 · 그 시드의 문턱 (유형 있는 위반 {int(SC.sum())} · 음성 {int(NEG.sum())} · 광고 문구 줄의 위반 {int((SC & AD).sum())})")
display(pd.DataFrame(_t).T)
print("시드별 (자기 문턱) — 위반 탐지 · 정답 유형 적중 · 음성 오판정:")
for name in RUNS:
    print(f"  {name}: " + "  /  ".join(f"시드 {s} → {v['det']} · {v['hit']} · {v['neg']}" for s, v in M[name]["per_seed"].items()))

# ── 표 2 · 유형별
_t = {}
for i, l in enumerate(CONF8):
    n = int(Y8[:, i].sum())
    _t[l] = {"dev 양성": f"{n}" + (" (측정 불가 n<30)" if n < 30 else ""),
             "Baseline v10 재현율": "—" if l not in b["type_tp"] else f"{b['type_tp'][l][0] / b['type_tp'][l][1] * 100:.1f}%"}
    for name in RUNS:
        s = M[name]["type"][l]
        _t[l][name] = f"R {s['tp'] / max(s['n'], 1) * 100:.1f}% · P {s['tp'] / max(s['pp'], 1) * 100:.1f}% · τ {RESULTS[name]['thresholds'][l]}"
print("\n■ 표 2 · 유형별 재현율 R · 정밀도 P · 문턱 τ")
display(pd.DataFrame(_t).T)

# ── 표 3 · 통과 기준 — 규약의 Baseline 값 대비
GATES, _t = {}, {}
for name in RUNS:
    R, m = RESULTS[name], M[name]
    d_pr = R["prauc6_mean"] - b_mean
    d_rec = m["det"] / m["scored"] - b["det"] / b["scored"]
    d_type = {l: m["type"][l]["tp"] / m["type"][l]["n"] - tp / n for l, (tp, n) in b["type_tp"].items() if m["type"][l]["n"] >= 30}
    worst = min(d_type, key=d_type.get) if d_type else None
    d_neg = m["neg"] - b["neg"]
    ok = {"B": d_pr >= -GATE["prauc_drop"], "G1": d_rec >= -GATE["recall_drop"] - 1e-12,
          "G2": (d_type[worst] >= -GATE["type_recall_drop"] - 1e-12) if worst else True, "G3": d_neg <= GATE["neg_flag_up"]}
    GATES[name] = {"fails": [k for k, v in ok.items() if not v], "d_prauc": d_pr, "d_recall": d_rec, "d_neg": d_neg, "worst_type": worst, "d_worst": d_type.get(worst)}
    _t[name] = {"B · PR-AUC 시드 평균": f"{R['prauc6_mean']:.4f} ({d_pr:+.4f})", "G1 · 위반 탐지율": f"{rate(m['det'], m['scored'])} ({d_rec * 100:+.1f}%p)",
                "G2 · 유형별 최대 하락": "—" if worst is None else f"{worst} {d_type[worst] * 100:+.1f}%p",
                "G3 · 음성 오판정": f"{m['neg']}/{m['neg_n']} ({d_neg:+d}행)", "걸린 기준": " · ".join(GATES[name]["fails"]) or "없음"}
print("\n■ 표 3 · 통과 기준 — Baseline(v10 · 규약 값) 대비")
display(pd.DataFrame(_t).T)
_m = M[BASE_RUN]
print(f"Baseline 재현 확인 — 시드별 PR-AUC {[round(RESULTS[BASE_RUN]['seeds'][s]['prauc6'], 4) for s in SEEDS]} (v10 {b['seed_prauc']}) · "
      f"위반 탐지 {_m['det']} (v10 {b['det']}) · 정답 유형 적중 {_m['hit']} (v10 {b['hit']}) · 음성 오판정 {_m['neg']} (v10 {b['neg']})")

# ── 표 4 · 합성 정상 문구 — 목표
_t = {}
for name in RUNS:
    d, a = SYN_M[name]["dev"], SYN_M[name]["amb"]
    _t[name] = {"합성 정상 dev — 후보가 선 행 ↓": rate(d["flag"], d["n"]), "조용한 행 (통과 가능) ↑": rate(d["quiet"], d["n"]),
                "카드 8 문턱에서 후보가 선 행": rate(d["flag_card8"], d["n"]),
                **{f"{c} — 후보 · 조용": f"{d['by_cat'][c][0]} · {d['by_cat'][c][1]} / {d['by_cat'][c][2]}" for c in SYN_CATS},
                "후보가 선 유형": " · ".join(f"{l} {v}" for l, v in d["by_type"].items() if v) or "없음",
                "가장 높은 확률 ÷ 문턱 (중앙값)": f"{d['max_ratio_median']:.2f}",
                "참고 — 애매 표시: 후보 · 조용": f"{a['flag']} · {a['quiet']} / {a['n']}"}
print(f"\n■ 표 4 · 합성 정상 문구 (학습에 넣지 않은 행) — 가장 좋은 시드 · 그 시드의 문턱")
display(pd.DataFrame(_t).T)
print("시드별 (자기 문턱) — 합성 정상 dev 에서 후보가 선 행 · 조용한 행:")
for name in RUNS:
    print(f"  {name}: " + "  /  ".join(f"시드 {s} → {v['dev']['flag']} · {v['dev']['quiet']}" for s, v in SYN_M[name]["per_seed"].items()))

VERDICT = {}
_b = SYN_M[BASE_RUN]["dev"]
for name in RUNS:
    if name == BASE_RUN:
        continue
    d = SYN_M[name]["dev"]
    limit = TARGET["syn_flag_ratio_max"] * _b["flag"]
    goal_met = _b["flag"] > 0 and d["flag"] <= limit + 1e-9         # Baseline 재현이 0행이면 줄었는지 잴 수 없다 — 목표 미달로 둔다
    fails = GATES[name]["fails"]
    verdict = ("탈락 (" + " · ".join(fails) + ")" + (" · 목표 충족" if goal_met else " · 목표 미달")) if fails else ("채택 후보" if goal_met else "통과 · 목표 미달")
    VERDICT[name] = {"verdict": verdict, "fails": fails, "goal_met": bool(goal_met), "syn_flag": d["flag"], "syn_flag_baseline": _b["flag"], "syn_flag_limit": limit, **GATES[name]}
    print(f"\n🔎 {name} — 합성 정상 dev 에서 후보가 선 행 {d['flag']}/{d['n']} · Baseline 재현 {_b['flag']}/{_b['n']} · 목표 {limit:.1f}행 이하 → {'충족' if goal_met else '미달'}")
    print(f"   미리 본 판정: {verdict}")
print("\n※ 미리 보는 값이다. 통과 기준의 판정은 공통 채점 스크립트로 하고(섹션 8), 채택 후보가 나와도 팀장 판정이 먼저다 — 골든 밖의 재료를 학습에 썼다.")
if SMOKE:
    print("⚠️ SMOKE 실행 — 위 수치는 의미 없다")

SUMMARY = {"notebook": NOTEBOOK, "golden_sha256": GOLDEN_SHA, "dev_sha": DEV_SHA, "dev_rows": len(dev_rows), "gate": GATE, "target": TARGET,
           "baseline": BASELINE, "smoke": SMOKE, "train": TRAIN_STATS, "verdict": VERDICT, "gates": GATES,
           "synthetic": {"sha256": SYN_SHA, "split": {"train": len(SYN_TRAIN), "dev": len(SYN_DEV), "amb": len(SYN_AMB)},
                         "hold_products": {c: sorted(p for cc, p in SYN_HOLD if cc == c) for c in SYN_CATS}, "hold_per_cat": SYN_HOLD_PER_CAT, "seed": SYN_SEED},
           "runs": {k: {"best_seed": RESULTS[k]["best_seed"], "prauc6_mean": RESULTS[k]["prauc6_mean"], "prauc6_std": RESULTS[k]["prauc6_std"],
                        "thresholds": RESULTS[k]["thresholds"], "use_synthetic_lawful": RUN_DEFS[k]["syn"],
                        "seeds": {str(s): {x: v[x] for x in ("prauc6", "prauc8", "best_epoch", "epochs_run", "curve", "minutes", "thresholds")}
                                  for s, v in RESULTS[k]["seeds"].items()},
                        "dev": {x: v for x, v in M[k].items() if x != "per_seed"},
                        "dev_by_seed": {str(s): v for s, v in M[k]["per_seed"].items()},
                        "syn": {x: v for x, v in SYN_M[k].items() if x != "per_seed"},
                        "syn_by_seed": {str(s): v for s, v in SYN_M[k]["per_seed"].items()}} for k in RUNS}}
json.dump(SUMMARY, open(f"{OUT_ROOT}/results_v10_card9.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2, default=float)
print("\n결과 폴더:", OUT_ROOT)
for f in sorted(os.listdir(OUT_ROOT)):
    if os.path.isfile(f"{OUT_ROOT}/{f}"):
        print(f"  {f:76s} {os.path.getsize(f'{OUT_ROOT}/{f}') / 1e6:8.1f} MB")

## 8. 다음 단계

**1) 받을 파일** — Drive `copylane_v10_card9/` 에서 **카드 9 의 시드별 dev CSV 3개 + `_scheme.zip`** 과 **두 실험의 합성 정상 CSV 6개**, `results_v10_card9.json` 을 받아 저장소 밖(예: `Downloads\copylane_v10\`)에 둔다. 모델 zip 은 채택 후보일 때만 받는다.

**2) 공통 채점 (로컬 · 통과 기준은 여기서 판정한다)**

```
python docs/psj/e2e_prototype/score_encoder_dev.py --probs <카드9_seed42_dev.csv> <카드9_seed43_dev.csv> <카드9_seed44_dev.csv> --scheme <카드9_scheme.zip> --baseline-probs <v10-합성O-이유O_dev행별확률.csv> --baseline-scheme <v10-합성O-이유O_scheme.zip> --baseline-seed-prauc 0.7167 0.7081 0.7165 --target "합성 정상 dev 에서 후보가 서는 비율이 Baseline 재현의 절반 이하" --out-md docs/psj/reports/v10_카드9_dev채점_20261008.md
```

**3) 무엇을 보고 고르나**

- 통과 기준에 하나라도 걸리면 **탈락**이다. 「문턱을 맞춘 비교」를 같이 읽어 탈락이 품질 탓인지 문턱이 옮겨 간 탓인지 적는다.
- 걸리지 않고 목표를 채우면 **채택 후보**다. 다만 이 재료는 골든 밖이고 사람이 판독하지 않았다 — 채택에는 팀장 판정이 든다.
- 목표 · 나누는 방법은 결과를 본 뒤 고치지 않는다. 음성 비율을 바꿔 보려면 새 카드를 쓴다.

**4) 올리지 않는 것** — 모델 zip · 행별 확률 CSV · 이 노트북의 실행 출력은 저장소에 올리지 않는다 (D-249). 노트북은 출력을 지우고 올린다.